In [ ]:
"""
Qube-Servo 3 — Task 4 Simulations (TODO Template)
(1) Analytic solution (ODE / inverse Laplace) — voltage -> omega(t), theta(t)
(2) Transfer-function simulation (python-control) — G_omega(s), G_theta(s)
(3) Qube-Servo run (simulator/hardware) — apply same step and log

Requires: numpy, matplotlib, control
Optional: pal.products.qube  (Quanser API) for part (3)
"""

import time
import numpy as np
import matplotlib.pyplot as plt
from pal.products.qube import QubeServo3
import control as ct

# ---- Qube-Servo 3 parameters (from datasheet) --------------------------------
Rm   = ...     # Ohm
Lm   = ...     # H (kept for 2nd-order extension; not used in 1st-order TF)
kt   = ...     # N*m/A
km   = ...     # V/(rad/s)
Jm   = ...     # kg*m^2
Jh   = ...     # kg*m^2
md   = ...     # kg (inertia disk)
rd   = ...     # m
Jd   = ...     # kg*m^2
Jeq  = ...     # total equivalent inertia at motor
b    = ...     # N*m*s/rad

# Convenience symbols students should derive/confirm
# TODO: define K and tau so that G(s) = K/(tau s + 1)
K     = ...
tau = ...

# ---- Step input ---------------------------------------------------------------
Vstep   = 1.0      # V (change amplitude as needed)
t_final = 4.0      # s
dt      = 0.002    # s
t = np.arange(0.0, t_final + dt, dt)
u = Vstep * np.ones_like(t)

In [ ]:
# ==============================================================================
# (1) ANALYTIC ODE SOLUTION
# For step input v(t)=Vstep*u(t), find closed forms:
#   omega(t) = ...
#   theta(t) = integral of omega(t)  (or inverse Laplace for Theta(s))
# Provide expressions using K, alpha, Jeq, and Vstep.
# ==============================================================================
omega_ode = ...    # rad/s, same shape as t
theta_ode = ...    # rad,   same shape as t

In [ ]:
# ==============================================================================
# (2) TRANSFER-FUNCTION SIMULATION (python-control)
# Use forced_response with the time grid t and input u = Vstep
# ==============================================================================
s = ct.TransferFunction.s

# TODO: define G_omega and G_theta using ct.TransferFunction
G_omega = ...
G_theta = ...

# TF responses (forced_response lets us specify custom input/time)
t_tf,  omega_tf = ct.forced_response(G_omega, T=t, U=u)   # rad/s
t_tf2, theta_tf = ct.forced_response(G_theta, T=t, U=u)   # rad

In [ ]:
# ---- Plots (one chart per figure) --------------------------------------------
# TF vs Analytic — speed
plt.figure(figsize=(8,5))
plt.plot(t_tf,  omega_tf,  label="TF speed (rad/s)")
plt.plot(t,     omega_ode, linestyle="--", label="Analytic ODE speed (rad/s)")
plt.xlabel("time (s)"); plt.ylabel("omega (rad/s)")
plt.title("Speed step response: TF vs Analytic")
plt.grid(True); plt.legend(); plt.tight_layout()
plt.show()

# TF vs Analytic — position
plt.figure(figsize=(8,5))
plt.plot(t_tf2, theta_tf,  label="TF position (rad)")
plt.plot(t,     theta_ode, linestyle="--", label="Analytic ODE position (rad)")
plt.xlabel("time (s)"); plt.ylabel("theta (rad)")
plt.title("Position step response: TF vs Analytic")
plt.grid(True); plt.legend(); plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# (3) QUBE-SERVO SIM/HARDWARE
# Use MotorPosition [rad] and MotorSpeed [rad/s] directly.
# ==============================================================================
SAFE_VOLTS = 3.0
def clip_v(v): return float(np.clip(v, -SAFE_VOLTS, SAFE_VOLTS))

qube_t, qube_v = [], []
qube_theta, qube_omega = [], []
with QubeServo3(hardware=0, pendulum=0) as qube:
    duration = t_final
    t0 = time.time()
    while True:
        t_now = time.time() - t0
        if t_now >= duration:
            break

        # TODO: read sensors, compute/clip voltage command, write it
        v_cmd = ...

        # TODO: Read angle and speed directly in SI units
        theta_rad = ... # [rad]
        omega_rad = ... # [rad/s]

        qube_t.append(t_now)
        qube_v.append(v_cmd)
        qube_theta.append(theta_rad)
        qube_omega.append(omega_rads)

        time.sleep(0.002)  # ~500 Hz loop

In [ ]:
# Qube traces (if logged)
if len(qube_t) > 0:
    plt.figure(figsize=(8,5))
    plt.plot(qube_t, qube_theta, label="Qube theta (rad)")
    plt.plot(t_tf2, theta_tf, linestyle="--", label="TF theta (rad)")
    plt.plot(t, theta_ode, linestyle=":", label="Analytic ODE theta (rad)")
    plt.xlabel("time (s)"); plt.ylabel("theta (rad)")
    plt.title("Qube (sim/hw) vs Models — Position")
    plt.grid(True); plt.legend(); plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(qube_t, qube_omega, label="Qube omega (rad/s)")
    plt.plot(t_tf,  omega_tf,  label="TF speed (rad/s)")
    plt.plot(t,     omega_ode, linestyle="--", label="Analytic ODE speed (rad/s)")
    plt.xlabel("time (s)"); plt.ylabel("omega (rad/s)")
    plt.title("Qube (sim/hw) vs Models — Speed")
    plt.grid(True); plt.legend(); plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(qube_t, qube_v, label="command voltage (V)")
    plt.xlabel("time (s)"); plt.ylabel("V")
    plt.title("Applied Step to Qube")
    plt.grid(True); plt.legend(); plt.tight_layout()
    plt.show()
else:
    print("[Qube] No device data to plot.")